In [283]:
import xarray as xr
from pathlib import Path
from dpyverification.datasources.fewsnetcdf import Preprocessor
from dpyverification.datasources.fewsnetcdf.main import FewsNetcdfKind

ImportError: cannot import name 'FewsNetcdfKind' from 'dpyverification.datasources.fewsnetcdf.main' (C:\Users\beunk\OneDrive - Stichting Deltares\Documents\GitHub\DPyVerification\src\dpyverification\datasources\fewsnetcdf\main.py)

In [29]:
path = Path(
    "c:/Users/beunk/OneDrive - Stichting Deltares/Documents/000 - Projects/"
    "tmp/markermeer_files",
)
preprocessor = Preprocessor(simobskind="sim", filter_variables=["waterlevel_model"])

rglob = "2024*.nc"
# ds = xr.open_mfdataset(
#     path.rglob(rglob),
#     combine="nested",
#     concat_dim="forecast_reference_time",
#     preprocess=preprocessor,
#     coords="minimal",
#     compat="override",
#     parallel=True,
#     chunks=None,
# )

ds2 = xr.open_mfdataset(
    path.rglob(rglob),
    combine="nested",
    concat_dim="analysis_time",
    # preprocess=preprocessor,
    coords="minimal",
    compat="override",
    parallel=True,
    chunks={"time": 600},
)

In [31]:
ds2.nbytes / (1024**2)

2797.0586366653442

In [55]:
time = []
real = []
stat = []
vars = []
for file in path.rglob(rglob):
    ds = xr.open_dataset(file)
    time.append(len(ds.time))
    real.append(len(ds.realization))
    stat.append(len(ds.stations))
    vars.append(len(ds.data_vars))   

In [56]:
set(time)

{721, 727, 733, 793}

In [57]:
set(real)

{24, 31, 51}

In [58]:
set(stat)

{1}

In [ ]:
Preprocessor.rename_dims_coords_to_internal(ds, simobskind="sim")

<xarray.Dataset> Size: 83kB
Dimensions:                  (time: 793, forecast_reference_time: 1,
                              realization: 24, station: 1)
Coordinates:
  * time                     (time) datetime64[ns] 6kB 2024-09-27 ... 2024-10...
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 8B 2024...
  * realization              (realization) int32 96B 0 1 2 3 4 ... 20 21 22 23
    lat                      (station) float64 8B ...
    lon                      (station) float64 8B ...
    y                        (station) float64 8B ...
    x                        (station) float64 8B ...
    z                        (station) float64 8B ...
    station_id               (station) |S64 64B ...
    station_name             (station) |S255 255B ...
Dimensions without coordinates: station
Data variables:
    waterlevel_model         (time, realization, station) float32 76kB ...
Attributes:
    Conventions:          CF-1.6
    coordinate_system:    WGS 1984
    featureType:          timeSeries
    time_coverage_start:  2024-09-27T00:00:00+0000
    time_coverage_end:    2024-10-02T12:00:00+0000
    geospatial_lon_min:   5.389019966125488
    geospatial_lon_max:   5.389019966125488
    geospatial_lat_min:   52.25282669067383
    geospatial_lat_max:   52.25282669067383

In [85]:
from dpyverification.datasources.fewsnetcdf.main import StandardCoord, StandardDim, FewsNetcdfCoord, FewsNetcdfDims, SimObsKind

class Preprocessor:
    """Used in xr.open_mfdataset(preprocess=preprocessor_instance)."""

    def __init__(
        self,
        simobskind: str,
        filter_variables: list[str] | None = None,
        filter_stations: list[str] | None = None,
        filter_forecast_periods: ForecastPeriods | None = None,
        *,
        transform_to_forecast_period_based_dataset: bool = False,
    ) -> None:
        """Set configuration for preprocessing."""
        self.simobskind = simobskind
        self.variables = filter_variables
        self.stations = filter_stations
        self.forecast_periods = filter_forecast_periods
        self.transform_to_forecast_period_based_dataset = transform_to_forecast_period_based_dataset

    @staticmethod
    def convert_byte_string_coord_to_utf8(
        dataset: xr.Dataset,
        coords: list[FewsNetcdfCoord],
    ) -> xr.Dataset:
        """Decode byte-strings to utf-8 strings for easy indexing."""
        for coord in coords:
            dataset[coord] = xr.DataArray(
                [  # type:ignore[misc]
                    v.decode("utf-8") if isinstance(v, bytes) else v  # type:ignore[misc]
                    for v in dataset[coord].to_numpy()  # type:ignore[misc]
                ],
                dims=dataset[coord].dims,
            )
        return dataset

    @staticmethod
    def rename_dims_coords_to_internal(dataset: xr.Dataset, simobskind: str) -> xr.Dataset:
        """Rename Delft-FEWS NetCDF file to internal SimObsDataset model."""
        rename_dict: dict[str, str] = {
            FewsNetcdfCoord.station_names: StandardCoord.station_name.name,
        }
        dataset = dataset.rename(rename_dict)

        # Rename stations dim to station
        rename_dict = {FewsNetcdfDims.stations: StandardDim.station}
        dataset = dataset.rename(rename_dict)

        # Set station_name variable as coordinate
        dataset = dataset.set_coords(StandardCoord.station_name.name)

        if simobskind == SimObsKind.OBS:
            return dataset

        # Simulations
        # Rename analysis_time
        rename_dict2 = {
            FewsNetcdfDims.analysis_time: StandardDim.forecast_reference_time,
        }
        return dataset.rename(rename_dict2)

    @staticmethod
    def filter_stations(dataset: xr.Dataset, stations: list[str]) -> xr.Dataset:
        """Filter stations from dataset."""
        swap_dict: dict[str, str] = {StandardDim.station: StandardCoord.station_id.name}
        dataset_swapped = dataset.swap_dims(swap_dict)
        sel_dict: dict[str, list[str]] = {StandardCoord.station_id.name: stations}
        dataset_swapped_selection = dataset_swapped.sel(sel_dict)
        swap_dict2: dict[str, str] = {
            StandardCoord.station_id.name: StandardDim.station,
        }
        return dataset_swapped_selection.swap_dims(
            swap_dict2,
        )

    @staticmethod
    def transform_to_forecast_period_dataset(dataset: xr.Dataset) -> xr.Dataset:
        """Transform forecast_reference_time dataset to forecast_period dataset."""
        forecast_period_coord = (
            dataset[StandardDim.time] - dataset[StandardDim.forecast_reference_time]
        ).astype(
            "timedelta64[ns]",
        )

        # Set forecast_period as coordinate on data variables
        for data_var in dataset.data_vars:
            da = dataset[data_var]
            da = da.expand_dims(
                {StandardDim.forecast_period: forecast_period_coord.size},
            )
            da = da.assign_coords(
                {StandardDim.forecast_period: forecast_period_coord.to_numpy().ravel()},  # type:ignore[misc]
            )
            dataset[data_var] = da

        # Drop forecast_reference_time
        return dataset.drop(StandardDim.forecast_reference_time)

    @staticmethod
    def filter_forecast_periods(
        dataset: xr.Dataset,
        forecast_periods: ForecastPeriods,
    ) -> xr.Dataset:
        """Filter only relevant lead times from dataset."""
        selection_dict: dict[str, list[np.timedelta64]] = {
            StandardCoord.forecast_period.name: forecast_periods.timedelta64,
        }
        return dataset.sel(selection_dict)

    def __call__(self, dataset: xr.Dataset) -> xr.Dataset:
        """Preprocess data."""
        # Convert station_id coordinate to utf-8 string
        dataset = Preprocessor.convert_byte_string_coord_to_utf8(
            dataset,
            coords=[FewsNetcdfCoord.station_id],
        )

        # Rename dims/coords to internal definition
        dataset = Preprocessor.rename_dims_coords_to_internal(
            dataset,
            simobskind=self.simobskind,
        )

        # Transform to forecast_period based, for memory efficiency
        if self.transform_to_forecast_period_based_dataset:
            dataset = Preprocessor.transform_to_forecast_period_dataset(dataset)

        # Filter variables
        if self.variables is not None:
            dataset = dataset[self.variables]

        # Filter only relevant forecast periods
        if self.forecast_periods is not None and self.transform_to_forecast_period_based_dataset:
            return Preprocessor.filter_forecast_periods(
                dataset,
                forecast_periods=self.forecast_periods,
            )
        return dataset

In [107]:
# New
import numpy as np
import xarray as xr
from dpyverification.datasources.fewsnetcdf.main import (
    StandardCoord,
    StandardDim,
    FewsNetcdfCoord,
    FewsNetcdfDims,
    SimObsKind,
    ForecastPeriods
)

class Preprocessor:
    """Used in xr.open_mfdataset(preprocess=preprocessor_instance)."""

    def __init__(
        self,
        simobskind: str,
        filter_variables: list[str] | None = None,
        filter_stations: list[str] | None = None,
        filter_forecast_periods: list[np.timedelta64] | None = None,
        *,
        transform_to_forecast_period_based_dataset: bool = False,
    ) -> None:
        self.simobskind = simobskind
        self.variables = filter_variables
        self.stations = filter_stations
        self.forecast_periods = filter_forecast_periods
        self.transform_to_forecast_period_based_dataset = transform_to_forecast_period_based_dataset

    @staticmethod
    def convert_byte_string_coord_to_utf8(dataset: xr.Dataset, coords: list[FewsNetcdfCoord]) -> xr.Dataset:
        for coord in coords:
            dataset[coord] = xr.DataArray(
                [v.decode("utf-8") if isinstance(v, bytes) else v for v in dataset[coord].to_numpy()],
                dims=dataset[coord].dims,
            )
        return dataset

    @staticmethod
    def rename_dims_coords_to_internal(dataset: xr.Dataset, simobskind: str) -> xr.Dataset:
        # Rename station coords/dims
        dataset = dataset.rename({FewsNetcdfCoord.station_names: StandardCoord.station_name.name})
        dataset = dataset.rename({FewsNetcdfDims.stations: StandardDim.station})
        dataset = dataset.set_coords(StandardCoord.station_name.name)

        if simobskind == SimObsKind.SIM:
            # Rename analysis_time for simulations
            dataset = dataset.rename({FewsNetcdfDims.analysis_time: StandardDim.forecast_reference_time})

        return dataset

    @staticmethod
    def filter_stations(dataset: xr.Dataset, stations: list[str]) -> xr.Dataset:
        swap_dict = {StandardDim.station: StandardCoord.station_id.name}
        dataset = dataset.swap_dims(swap_dict)
        dataset = dataset.sel({StandardCoord.station_id.name: stations})
        return dataset.swap_dims({StandardCoord.station_id.name: StandardDim.station})

    @staticmethod
    def transform_to_forecast_period_dataset(dataset: xr.Dataset) -> xr.Dataset:
        """
        Transform dataset so that 'forecast_period' is a dimension and 'time' remains
        a coordinate for explicit timestamps. This allows slicing by forecast_period
        while keeping the timestamp available.
        """
        # Compute forecast_period relative to forecast_reference_time
        forecast_period_coord = (dataset[StandardDim.time] - dataset[StandardDim.forecast_reference_time]).astype("timedelta64[ns]")

        # Assign forecast_period as a new coordinate along the time dimension
        dataset = dataset.assign_coords({StandardDim.forecast_period: ("time", forecast_period_coord.to_numpy().ravel())})

        # Swap time dimension to forecast_period dimension
        dataset = dataset.swap_dims({StandardDim.time: StandardDim.forecast_period})

        # Optionally keep time as a coordinate
        # time is now accessible as dataset.time
        # Drop forecast_reference_time if not needed
        # dataset = dataset.drop_vars(StandardDim.forecast_reference_time)

        return dataset


    @staticmethod
    def filter_forecast_periods(dataset: xr.Dataset, forecast_periods: ForecastPeriods) -> xr.Dataset:
        """Select only requested lead times, preserving time coordinates."""
        return dataset.sel({StandardDim.forecast_period: forecast_periods.timedelta64})

    def __call__(self, dataset: xr.Dataset) -> xr.Dataset:
        # Decode byte-string coords
        dataset = self.convert_byte_string_coord_to_utf8(dataset, coords=[FewsNetcdfCoord.station_id])

        # Rename dims/coords to internal definitions
        dataset = self.rename_dims_coords_to_internal(dataset, simobskind=self.simobskind)

        # Transform to forecast_period-based dataset
        if self.transform_to_forecast_period_based_dataset:
            dataset = self.transform_to_forecast_period_dataset(dataset)

        # Filter variables
        if self.variables is not None:
            dataset = dataset[self.variables]

        # Filter stations
        if self.stations is not None:
            dataset = self.filter_stations(dataset, self.stations)

        # Filter forecast_periods (lead times)
        if self.forecast_periods is not None and self.transform_to_forecast_period_based_dataset:
            dataset = self.filter_forecast_periods(dataset, self.forecast_periods)

        return dataset


In [130]:
from dpyverification.datasources.fewsnetcdf.main import FewsNetcdfCoord, ForecastPeriods

preprocessor = Preprocessor(
    simobskind="sim",
    filter_variables=["waterlevel_model"],
    filter_forecast_periods=ForecastPeriods(unit="h", values=[12, 24, 36, 48]),
    transform_to_forecast_period_based_dataset=True,
)

new_ds = preprocessor(ds)

In [111]:
new_ds.isel(forecast_period=0, realization=0, station=0).to_pandas()

waterlevel_model   -0.437
dtype: float64

In [126]:
new_ds["waterlevel_model"].isel(station=0, realization=0, forecast_period=0).to_pandas()

array(-0.437, dtype=float32)

In [128]:
new_ds.nbytes / (1024**2)

0.0008535385131835938

In [129]:
ds.nbytes / (1024**2)

0.0790853500366211

In [131]:
new_ds

<xarray.Dataset> Size: 895B
Dimensions:           (forecast_period: 4, realization: 24, station: 1)
Coordinates:
    time              (forecast_period) datetime64[ns] 32B 2024-09-27T12:00:0...
  * realization       (realization) int32 96B 0 1 2 3 4 5 ... 18 19 20 21 22 23
    lat               (station) float64 8B ...
    lon               (station) float64 8B ...
    y                 (station) float64 8B ...
    x                 (station) float64 8B ...
    z                 (station) float64 8B ...
    station_id        (station) <U14 56B 'GemaalVeendijk'
    station_name      (station) |S255 255B ...
  * forecast_period   (forecast_period) timedelta64[ns] 32B 0 days 12:00:00 ....
Dimensions without coordinates: station
Data variables:
    waterlevel_model  (forecast_period, realization, station) float32 384B ...
Attributes:
    Conventions:          CF-1.6
    coordinate_system:    WGS 1984
    featureType:          timeSeries
    time_coverage_start:  2024-09-27T00:00:00+0000
    time_coverage_end:    2024-10-02T12:00:00+0000
    geospatial_lon_min:   5.389019966125488
    geospatial_lon_max:   5.389019966125488
    geospatial_lat_min:   52.25282669067383
    geospatial_lat_max:   52.25282669067383

In [133]:
new_ds2 = new_ds.copy(deep=True)

In [142]:
new_ds2 = new_ds2.assign_coords({"time": (new_ds2.time + np.timedelta64(1, "D"))})

In [264]:
new_ds2.time

<xarray.DataArray 'time' (forecast_period: 4)> Size: 32B
array(['2024-09-28T12:00:00.000000000', '2024-09-29T00:00:00.000000000',
       '2024-09-29T12:00:00.000000000', '2024-09-30T00:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
    time             (forecast_period) datetime64[ns] 32B 2024-09-28T12:00:00...
  * forecast_period  (forecast_period) timedelta64[ns] 32B 0 days 12:00:00 .....

In [203]:
# New
import numpy as np
import xarray as xr
from dpyverification.datasources.fewsnetcdf.main import (
    StandardCoord,
    StandardDim,
    FewsNetcdfCoord,
    FewsNetcdfDims,
    SimObsKind,
    ForecastPeriods
)

class Preprocessor:
    """Used in xr.open_mfdataset(preprocess=preprocessor_instance)."""

    def __init__(
        self,
        simobskind: str,
        filter_variables: list[str] | None = None,
        filter_stations: list[str] | None = None,
        filter_forecast_periods: list[np.timedelta64] | None = None,
        *,
        transform_to_forecast_period_based_dataset: bool = False,
    ) -> None:
        self.simobskind = simobskind
        self.variables = filter_variables
        self.stations = filter_stations
        self.forecast_periods = filter_forecast_periods
        self.transform_to_forecast_period_based_dataset = transform_to_forecast_period_based_dataset

    @staticmethod
    def convert_byte_string_coord_to_utf8(dataset: xr.Dataset, coords: list[FewsNetcdfCoord]) -> xr.Dataset:
        for coord in coords:
            dataset[coord] = xr.DataArray(
                [v.decode("utf-8") if isinstance(v, bytes) else v for v in dataset[coord].to_numpy()],
                dims=dataset[coord].dims,
            )
        return dataset

    @staticmethod
    def rename_dims_coords_to_internal(dataset: xr.Dataset, simobskind: str) -> xr.Dataset:
        # Rename station coords/dims
        dataset = dataset.rename({FewsNetcdfCoord.station_names: StandardCoord.station_name.name})
        dataset = dataset.rename({FewsNetcdfDims.stations: StandardDim.station})
        dataset = dataset.set_coords(StandardCoord.station_name.name)

        if simobskind == SimObsKind.SIM:
            # Rename analysis_time for simulations
            dataset = dataset.rename({FewsNetcdfDims.analysis_time: StandardDim.forecast_reference_time})

        return dataset

    @staticmethod
    def filter_stations(dataset: xr.Dataset, stations: list[str]) -> xr.Dataset:
        swap_dict = {StandardDim.station: StandardCoord.station_id.name}
        dataset = dataset.swap_dims(swap_dict)
        dataset = dataset.sel({StandardCoord.station_id.name: stations})
        return dataset.swap_dims({StandardCoord.station_id.name: StandardDim.station})

    @staticmethod
    def transform_to_forecast_period_dataset(dataset: xr.Dataset) -> xr.Dataset:
        """
        Transform dataset so that 'forecast_period' is a dimension and 'time' remains
        a coordinate for explicit timestamps. This allows slicing by forecast_period
        while keeping the timestamp available.
        """
        # Compute forecast_period relative to forecast_reference_time
        forecast_period_coord = (dataset[StandardDim.time] - dataset[StandardDim.forecast_reference_time]).astype("timedelta64[ns]")

        # Assign forecast_period as a new coordinate along the time dimension
        dataset = dataset.assign_coords({StandardDim.forecast_period: ("time", forecast_period_coord.to_numpy().ravel())})

        # Swap time dimension to forecast_period dimension
        dataset = dataset.swap_dims({StandardDim.time: StandardDim.forecast_period})

        # Optionally keep time as a coordinate
        # time is now accessible as dataset.time
        # Drop forecast_reference_time if not needed
        # dataset = dataset.drop_vars(StandardDim.forecast_reference_time)

        return dataset


    @staticmethod
    def filter_forecast_periods(dataset: xr.Dataset, forecast_periods: ForecastPeriods) -> xr.Dataset:
        """Select only requested lead times, preserving time coordinates."""
        return dataset.sel({StandardDim.forecast_period: forecast_periods.timedelta64})

    def __call__(self, dataset: xr.Dataset) -> xr.Dataset:
        # Decode byte-string coords
        dataset = self.convert_byte_string_coord_to_utf8(dataset, coords=[FewsNetcdfCoord.station_id])

        # Rename dims/coords to internal definitions
        dataset = self.rename_dims_coords_to_internal(dataset, simobskind=self.simobskind)

        # Transform to forecast_period-based dataset
        if self.transform_to_forecast_period_based_dataset:
            dataset = self.transform_to_forecast_period_dataset(dataset)

        # Filter variables
        if self.variables is not None:
            dataset = dataset[self.variables]

        # Filter stations
        if self.stations is not None:
            dataset = self.filter_stations(dataset, self.stations)

        # Filter forecast_periods (lead times)
        if self.forecast_periods is not None and self.transform_to_forecast_period_based_dataset:
            dataset = self.filter_forecast_periods(dataset, self.forecast_periods)

        return dataset


In [278]:
import numpy as np
import pandas as pd
import xarray as xr

# Parameters
n_ref = 3        # number of forecast reference times
n_period = 4     # number of lead times (forecast_periods)
n_station = 2    # number of stations

# Create forecast reference times
forecast_reference_times = pd.date_range("2025-08-13 00:00", periods=n_ref, freq="6h")

# Create forecast periods (lead times in hours)
forecast_periods = np.array([6, 12, 18, 24])  # hours

# Create station IDs
stations = ["station_1", "station_2"]

# Generate dummy forecast data
data = np.random.rand(n_period, n_ref, n_station)

# Generate 2D time coordinate
time = np.array([
    [ref + pd.Timedelta(hours=lead) for ref in forecast_reference_times]
    for lead in forecast_periods
])

# Create xarray Dataset
ds = xr.Dataset(
    {
        "forecast": (("forecast_period", "forecast_reference_time", "station"), data)
    },
    coords={
        "forecast_reference_time": forecast_reference_times,
        "forecast_period": forecast_periods,
        "station": stations,
        "time": (("forecast_period", "forecast_reference_time"), time)
    }
)

ds

<xarray.Dataset> Size: 416B
Dimensions:                  (forecast_period: 4, forecast_reference_time: 3,
                              station: 2)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 24B 202...
  * forecast_period          (forecast_period) int64 32B 6 12 18 24
  * station                  (station) <U9 72B 'station_1' 'station_2'
    time                     (forecast_period, forecast_reference_time) datetime64[ns] 96B ...
Data variables:
    forecast                 (forecast_period, forecast_reference_time, station) float64 192B ...

In [279]:
ds.isel(forecast_period=0).swap_dims({"forecast_reference_time":"time"})

<xarray.Dataset> Size: 176B
Dimensions:                  (time: 3, station: 2)
Coordinates:
    forecast_reference_time  (time) datetime64[ns] 24B 2025-08-13 ... 2025-08...
    forecast_period          int64 8B 6
  * station                  (station) <U9 72B 'station_1' 'station_2'
  * time                     (time) datetime64[ns] 24B 2025-08-13T06:00:00 .....
Data variables:
    forecast                 (time, station) float64 48B 0.2611 ... 0.2577

In [382]:
# Post Cees implementation

# New
import numpy as np
import xarray as xr
from dpyverification.datasources.fewsnetcdf.main import (
    StandardCoord,
    StandardDim,
    FewsNetcdfCoord,
    FewsNetcdfDims,
    SimObsKind,
    ForecastPeriods,
)

from enum import StrEnum

class FewsNetcdfKind(StrEnum):
    """List of kinds of FEWS NetCDFs."""

    observation = "observation"
    one_full_simulation = "one_full_simulation"
    simulation_for_one_forecast_period = "simulation_for_one_forecast_period"

class Preprocessor:
    """Used in xr.open_mfdataset(preprocess=preprocessor_instance)."""

    def __init__(
        self,
        fews_netcdf_kind: FewsNetcdfKind,
        filter_variables: list[str] | None = None,
        filter_stations: list[str] | None = None,
        filter_forecast_periods: list[np.timedelta64] | None = None,
    ) -> None:
        self.fews_netcdf_kind = fews_netcdf_kind
        self.variables = filter_variables
        self.stations = filter_stations
        self.forecast_periods = filter_forecast_periods

    @staticmethod
    def convert_byte_string_coord_to_utf8(dataset: xr.Dataset, coords: list[FewsNetcdfCoord]) -> xr.Dataset:
        for coord in coords:
            dataset[coord] = xr.DataArray(
                [v.decode("utf-8") if isinstance(v, bytes) else v for v in dataset[coord].to_numpy()],
                dims=dataset[coord].dims,
            )
        return dataset

    @staticmethod
    def rename_dims_coords_to_internal(dataset: xr.Dataset, fews_netcdf_kind: str) -> xr.Dataset:
        # Rename station coords/dims
        dataset = dataset.rename({FewsNetcdfCoord.station_names: StandardCoord.station_name.name})
        dataset = dataset.rename({FewsNetcdfDims.stations: StandardDim.station})
        dataset = dataset.set_coords(StandardCoord.station_name.name)

        if fews_netcdf_kind == FewsNetcdfKind.one_full_simulation:
            # Rename analysis_time for simulations
            dataset = dataset.rename({FewsNetcdfDims.analysis_time: StandardDim.forecast_reference_time})

        return dataset

    @staticmethod
    def filter_stations(dataset: xr.Dataset, stations: list[str]) -> xr.Dataset:
        swap_dict = {StandardDim.station: StandardCoord.station_id.name}
        dataset = dataset.swap_dims(swap_dict)
        dataset = dataset.sel({StandardCoord.station_id.name: stations})
        return dataset.swap_dims({StandardCoord.station_id.name: StandardDim.station})

    @staticmethod
    def transform_to_forecast_period_dataset(dataset: xr.Dataset) -> xr.Dataset:
        """
        Transform dataset so that 'forecast_period' is a dimension and 'time' remains
        a coordinate for explicit timestamps. This allows slicing by forecast_period
        while keeping the timestamp available.
        """
        for data_var in dataset.data_vars:
            da = dataset[data_var]

            da_list = []
            for forecast_period in da["forecast_period"]:
                da_fp = da.sel({"forecast_period": forecast_period})
                da_fp = da_fp.expand_dims({"forecast_period": 1})
                da_fp = da_fp.assign_coords({"forecast_period": ("forecast_period", forecast_period.to_numpy().reshape(1))})
                da_fp = da_fp.swap_dims({"forecast_reference_time": "time"})
                da_list.append(da_fp)
        
        return xr.merge(da_list)

    @staticmethod
    def transform_full_simulation_to_full_info_sim_dataset(dataset: xr.Dataset, filter_forecast_periods: ForecastPeriods):
        """Transform the dataset to full-information dataset."""

        forecast_periods = (dataset.time - dataset.forecast_reference_time).to_numpy().ravel()
        ds = dataset.assign_coords({"forecast_period": ("time", forecast_periods)})
        
        ds = ds.swap_dims({"time": "forecast_period"}).drop_vars("time")
            
        # Re-compute time as 2d matrix along forecast_period and forecast_reference_time
        time_index_2d = (ds["forecast_reference_time"] + ds["forecast_period"]).to_numpy().swapaxes(0,1)
        
        # Now assign time
        ds = ds.assign_coords({"time": (("forecast_period", "forecast_reference_time"), time_index_2d)})
        

        # Filter data variables explicitly
        for data_var in ds.data_vars:
            ds[data_var] = (
                ds[data_var]
                .expand_dims({"forecast_reference_time": ds["forecast_reference_time"]})
            )
    
        # Filter the relevant forecast_periods to maximize memory efficiency 
        selector = {"forecast_period": filter_forecast_periods}
        ds = ds.sel(selector)

        return ds

    def __call__(self, dataset: xr.Dataset) -> xr.Dataset:
        # Decode byte-string coords
        dataset = self.convert_byte_string_coord_to_utf8(dataset, coords=[FewsNetcdfCoord.station_id])

        # Rename dims/coords to internal definitions
        dataset = self.rename_dims_coords_to_internal(dataset, fews_netcdf_kind=self.fews_netcdf_kind)

        # Transform to full info sim dataset 
        if self.fews_netcdf_kind == FewsNetcdfKind.one_full_simulation:
            dataset = self.transform_full_simulation_to_full_info_sim_dataset(dataset, self.forecast_periods)

        # Filter variables
        if self.variables is not None:
            dataset = dataset[self.variables]

        # Filter stations
        if self.stations is not None:
            dataset = self.filter_stations(dataset, self.stations)

        return dataset


In [383]:
dataset = xr.open_dataset(r"c:\Users\beunk\OneDrive - Stichting Deltares\Documents\000 - Projects\tmp\markermeer_files\2024010108.nc")

preprocessor = Preprocessor(fews_netcdf_kind=FewsNetcdfKind.one_full_simulation, 
                            filter_variables=["waterlevel_model"], 
                            filter_forecast_periods=ForecastPeriods(unit="h", values=[12, 24, 36, 48]).timedelta64)

dss = preprocessor(dataset)
dss.nbytes / (1024**2)

0.0013761520385742188

In [366]:
full_ds = xr.open_mfdataset(
    path.rglob("*.nc"),
    combine="nested",
    concat_dim="forecast_reference_time",
    preprocess=preprocessor,
    coords="minimal",
    compat="override",
    parallel=True,
)

In [367]:
full_ds.load()

<xarray.Dataset> Size: 342kB
Dimensions:                  (forecast_reference_time: 399, forecast_period: 4,
                              realization: 51, station: 1)
Coordinates:
  * realization              (realization) int32 204B 0 1 2 3 4 ... 47 48 49 50
  * forecast_period          (forecast_period) timedelta64[ns] 32B 0 days 12:...
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 3kB 202...
    lat                      (station) float64 8B 52.25
    lon                      (station) float64 8B 5.389
    y                        (station) float64 8B 52.25
    x                        (station) float64 8B 5.389
    z                        (station) float64 8B 0.0
    station_id               (station) <U14 56B 'GemaalVeendijk'
    station_name             (station) |S255 255B b'Veendijk Gemaal Eemmeer'
    time                     (forecast_period, forecast_reference_time) datetime64[ns] 13kB ...
Dimensions without coordinates: station
Data variables:
    waterlevel_model         (forecast_reference_time, forecast_period, realization, station) float32 326kB ...
Attributes:
    Conventions:          CF-1.6
    coordinate_system:    WGS 1984
    featureType:          timeSeries
    time_coverage_start:  2024-01-01T08:00:00+0000
    time_coverage_end:    2024-01-06T09:00:00+0000
    geospatial_lon_min:   5.389019966125488
    geospatial_lon_max:   5.389019966125488
    geospatial_lat_min:   52.25282669067383
    geospatial_lat_max:   52.25282669067383

In [384]:
def transform_to_forecast_period_dataset(dataset: xr.Dataset) -> xr.Dataset:
    """
    Transform dataset so that 'forecast_period' is a dimension and 'time' remains
    a coordinate for explicit timestamps. This allows slicing by forecast_period
    while keeping the timestamp available.
    """
    for data_var in dataset.data_vars:
        da = dataset[data_var]

        da_list = []
        for forecast_period in da["forecast_period"]:
            da_fp = da.sel({"forecast_period": forecast_period})
            da_fp = da_fp.expand_dims({"forecast_period": 1})
            da_fp = da_fp.assign_coords({"forecast_period": ("forecast_period", forecast_period.to_numpy().reshape(1))})
            da_fp = da_fp.swap_dims({"forecast_reference_time": "time"})
            da_list.append(da_fp)
    
    return xr.combine_by_coords(da_list)

ds = transform_to_forecast_period_dataset(full_ds)
ds

<xarray.Dataset> Size: 399kB
Dimensions:                  (forecast_period: 4, time: 465, realization: 51,
                              station: 1)
Coordinates:
  * realization              (realization) int32 204B 0 1 2 3 4 ... 47 48 49 50
  * time                     (time) datetime64[ns] 4kB 2024-01-01T20:00:00 .....
  * forecast_period          (forecast_period) timedelta64[ns] 32B 0 days 12:...
    forecast_reference_time  (forecast_period, time) datetime64[ns] 15kB 2024...
    lat                      (station) float64 8B 52.25
    lon                      (station) float64 8B 5.389
    y                        (station) float64 8B 52.25
    x                        (station) float64 8B 5.389
    z                        (station) float64 8B 0.0
    station_id               (station) <U14 56B 'GemaalVeendijk'
    station_name             (station) |S255 255B b'Veendijk Gemaal Eemmeer'
Dimensions without coordinates: station
Data variables:
    waterlevel_model         (forecast_period, time, realization, station) float32 379kB ...